# Colab T4 추가 측정: 256×256 (Study Guide 셀 4 시간 근거용)

**런타임 → 런타임 유형 변경 → T4 GPU** 후 **런타임 → 모두 실행**. 끝나면 .ipynb로 다운로드한다. 전체 약 15분.

Study Guide 셀 4 예시("T4에서 256×256 400장의 1에폭이 5.9초면 842장은 12초")와 같은 조건을 직접 잰다.
템플릿과 같은 데이터 처리(좌우 뒤집기, 정답 float 0/1, VOC 255 무시)와 과제1 U-Net을 쓴다.

In [8]:
QUICK = False   # True면 에폭을 1로 줄여 빠르게 동작만 확인

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

import time, random, copy, json
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
print("device:", device, "|", GPU_NAME, "| torch", torch.__version__)
pd.set_option("display.max_colwidth", 60); pd.set_option("display.width", 200)

def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()

device: cuda | Tesla T4 | torch 2.11.0+cu130


In [9]:
# 과제1 B와 같은 U-Net (out_ch=1이면 sigmoid, 아니면 로짓)
def double_conv(i_, o_, bn=True):
    L = []
    for i, o in [(i_, o_), (o_, o_)]:
        L.append(nn.Conv2d(i, o, 3, padding=1))
        if bn: L.append(nn.BatchNorm2d(o))
        L.append(nn.ReLU(inplace=True))
    return nn.Sequential(*L)

class UNet(nn.Module):
    def __init__(self, base=16, n_pool=3, skip=True, bn=True, out_ch=1):
        super().__init__(); self.skip = skip; self.out_ch = out_ch
        chs = [base * 2 ** k for k in range(n_pool + 1)]
        self.encoders = nn.ModuleList([double_conv(3, chs[0], bn)] + [double_conv(chs[k-1], chs[k], bn) for k in range(1, n_pool + 1)])
        self.pool = nn.MaxPool2d(2)
        self.decoders = nn.ModuleList([double_conv(chs[k+1] + (chs[k] if skip else 0), chs[k], bn) for k in reversed(range(n_pool))])
        self.final = nn.Conv2d(base, out_ch, 1)
    def forward(self, x):
        skips = []
        for k, e in enumerate(self.encoders):
            if k > 0: x = self.pool(x)
            x = e(x); skips.append(x)
        x = skips.pop()
        for d in self.decoders:
            x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            s = skips.pop()
            x = d(torch.cat([s, x], 1) if self.skip else x)
        out = self.final(x)
        return torch.sigmoid(out) if self.out_ch == 1 else out

def n_params(m):
    return sum(p.numel() for p in m.parameters())

def infer_ms(model, size):
    model.eval(); x = torch.rand(1, 3, size, size, device=device)
    with torch.no_grad():
        for _ in range(10): model(x)
        sync(); t = time.time()
        for _ in range(50): model(x)
        sync()
    return (time.time() - t) / 50 * 1000

In [10]:
IGNORE = 255
class SegDataset(Dataset):
    def __init__(self, raw, option, n=None, size=256, augment=False, voc_target=(15,)):
        n = len(raw) if n is None else min(n, len(raw)); self.augment = augment
        self.images, self.masks = [], []
        for i in range(n):
            img, m = raw[i]
            img = np.array(img.convert("RGB").resize((size, size), Image.BILINEAR), dtype=np.uint8)
            m = np.array(m.resize((size, size), Image.NEAREST)).astype(np.int64)
            if option == "pet":
                lab = (m == 1).astype(np.uint8)
            else:
                lab = np.isin(m, voc_target).astype(np.uint8); lab[m == 255] = IGNORE
            self.images.append(img); self.masks.append(lab)
    def __len__(self): return len(self.images)
    def __getitem__(self, i):
        x = torch.tensor(self.images[i], dtype=torch.float32).permute(2, 0, 1) / 255.0
        y = torch.tensor(self.masks[i], dtype=torch.float32)
        if self.augment and random.random() < 0.5: x, y = x.flip(-1), y.flip(-1)
        return x, y

def run(name, option, n_train, size, n_pool, epochs, pos_weight=None, base=16):
    torch.manual_seed(0); random.seed(0); np.random.seed(0)
    key = (option, size)
    tr_full, te = CACHE[key]
    tr = torch.utils.data.Subset(tr_full, list(range(min(n_train, len(tr_full))))) if n_train else tr_full
    masks = np.stack([tr_full.masks[i] for i in (range(len(tr)) if n_train else range(len(tr_full)))])
    valid = masks != IGNORE; pos = float((masks == 1).sum() / valid.sum())
    pw = (np.sqrt((1 - pos) / pos) if pos_weight == "auto" else pos_weight)
    bce = nn.BCELoss(reduction="none")
    def crit(p, t):
        v = (t != IGNORE).float(); tt = t.clamp(max=1)
        w = v if pw is None else v * torch.where(tt == 1, torch.tensor(float(pw), device=device), torch.tensor(1.0, device=device))
        return (bce(p, tt) * w).sum() / v.sum().clamp(min=1)
    model = UNet(base, n_pool).to(device); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    tl = DataLoader(tr, batch_size=16, shuffle=True); vl = DataLoader(te, batch_size=16)
    secs, ious, fps = [], [], None
    E = 1 if QUICK else epochs
    for ep in range(E):
        model.train(); sync(); t = time.time()
        for x, y in tl:
            x, y = x.to(device), y.to(device); opt.zero_grad(); loss = crit(model(x).squeeze(1), y); loss.backward(); opt.step()
        sync(); secs.append(time.time() - t)
        model.eval(); il, fl = [], []
        with torch.no_grad():
            for x, y in vl:
                x, y = x.to(device), y.to(device); pr = model(x).squeeze(1) > 0.5
                for p, t in zip(pr, y):
                    v = t != IGNORE; t1 = (t == 1) & v; p1 = p & v
                    if t1.sum() == 0: fl.append((p1.sum() / v.sum()).item())
                    else: il.append(((p1 & t1).sum() / (p1 | t1).sum()).item())
        ious.append(float(np.mean(il))); fps = float(np.mean(fl)) if fl else None
        print(f"  [{name}] epoch {ep + 1}: {secs[-1]:.1f}s, IoU {ious[-1]:.3f}")
    return {"설정": name, "입력": f"{size}x{size}", "학습 장 수": len(tr), "풀링": n_pool, "파라미터 수": n_params(model),
            "1에폭 시간(초)": round(secs[0], 1), "에폭당 평균(초, 1에폭 제외)": round(float(np.mean(secs[1:])) if len(secs) > 1 else secs[0], 1),
            "에폭": E, "최종 IoU": round(ious[-1], 3), "최고 IoU(에폭)": f"{max(ious):.3f} ({int(np.argmax(ious)) + 1})",
            "대상 없는 사진 오탐": None if fps is None else f"{fps:.2%}", "장당 추론(ms)": round(infer_ms(model, size), 2)}

CACHE = {}
t = time.time()
rtr = datasets.OxfordIIITPet("data", split="trainval", target_types="segmentation", download=True)
rte = datasets.OxfordIIITPet("data", split="test", target_types="segmentation", download=True)
CACHE[("pet", 256)] = (SegDataset(rtr, "pet", augment=True), SegDataset(rte, "pet", n=500))
print(f"Pet 256 전처리 {time.time() - t:.0f}초")
rows = []
rows.append(run("Pet 256, 400장", "pet", 400, 256, 3, 5))
rows.append(run("Pet 256, 842장", "pet", 842, 256, 3, 5))
rows.append(run("Pet 256, 전부", "pet", None, 256, 3, 5))
rows.append(run("Pet 256, 전부, 풀링 4번", "pet", None, 256, 4, 5))
df_a = pd.DataFrame(rows); print(f"=== Pet 256 ({GPU_NAME}) ==="); display(df_a)

Pet 256 전처리 26초
  [Pet 256, 400장] epoch 1: 4.8s, IoU 0.283
  [Pet 256, 400장] epoch 2: 4.7s, IoU 0.380
  [Pet 256, 400장] epoch 3: 4.8s, IoU 0.429
  [Pet 256, 400장] epoch 4: 4.8s, IoU 0.445
  [Pet 256, 400장] epoch 5: 4.8s, IoU 0.338
  [Pet 256, 842장] epoch 1: 10.1s, IoU 0.395
  [Pet 256, 842장] epoch 2: 9.9s, IoU 0.477
  [Pet 256, 842장] epoch 3: 9.9s, IoU 0.462
  [Pet 256, 842장] epoch 4: 9.9s, IoU 0.498
  [Pet 256, 842장] epoch 5: 10.0s, IoU 0.506
  [Pet 256, 전부] epoch 1: 43.5s, IoU 0.466
  [Pet 256, 전부] epoch 2: 43.4s, IoU 0.567
  [Pet 256, 전부] epoch 3: 43.4s, IoU 0.575
  [Pet 256, 전부] epoch 4: 43.4s, IoU 0.380
  [Pet 256, 전부] epoch 5: 43.5s, IoU 0.642
  [Pet 256, 전부, 풀링 4번] epoch 1: 46.6s, IoU 0.438
  [Pet 256, 전부, 풀링 4번] epoch 2: 46.6s, IoU 0.559
  [Pet 256, 전부, 풀링 4번] epoch 3: 46.6s, IoU 0.594
  [Pet 256, 전부, 풀링 4번] epoch 4: 46.6s, IoU 0.608
  [Pet 256, 전부, 풀링 4번] epoch 5: 46.6s, IoU 0.670
=== Pet 256 (Tesla T4) ===


,설정,입력,학습 장 수,풀링,파라미터 수,1에폭 시간(초),"에폭당 평균(초, 1에폭 제외)",에폭,최종 IoU,최고 IoU(에폭),대상 없는 사진 오탐,장당 추론(ms)
0,"Pet 256, 400장",256x256,400,3,488705,4.8,4.8,5,0.338,0.445 (4),None,5.41
1,"Pet 256, 842장",256x256,842,3,488705,10.1,9.9,5,0.506,0.506 (5),None,5.38
2,"Pet 256, 전부",256x256,3680,3,488705,43.5,43.4,5,0.642,0.642 (5),None,5.31
3,"Pet 256, 전부, 풀링 4번",256x256,3680,4,1965569,46.6,46.6,5,0.670,0.670 (5),None,6.55


In [11]:
t = time.time()
vtr = datasets.VOCSegmentation("data", year="2012", image_set="train", download=True)
vte = datasets.VOCSegmentation("data", year="2012", image_set="val", download=True)
CACHE[("voc", 256)] = (SegDataset(vtr, "voc", augment=True), SegDataset(vte, "voc", n=500))
print(f"VOC 256 내려받기·전처리 {time.time() - t:.0f}초")
rows_b = [run("VOC 사람 256, pos_weight auto", "voc", None, 256, 3, 10, pos_weight="auto")]
df_b = pd.DataFrame(rows_b); print(f"=== VOC 256 ({GPU_NAME}) ==="); display(df_b)

100%|██████████| 2.00G/2.00G [02:06<00:00, 15.8MB/s]


VOC 256 내려받기·전처리 190초
  [VOC 사람 256, pos_weight auto] epoch 1: 17.1s, IoU 0.104
  [VOC 사람 256, pos_weight auto] epoch 2: 17.5s, IoU 0.222
  [VOC 사람 256, pos_weight auto] epoch 3: 17.4s, IoU 0.156
  [VOC 사람 256, pos_weight auto] epoch 4: 17.1s, IoU 0.263
  [VOC 사람 256, pos_weight auto] epoch 5: 17.1s, IoU 0.170
  [VOC 사람 256, pos_weight auto] epoch 6: 17.3s, IoU 0.244
  [VOC 사람 256, pos_weight auto] epoch 7: 17.3s, IoU 0.149
  [VOC 사람 256, pos_weight auto] epoch 8: 17.2s, IoU 0.217
  [VOC 사람 256, pos_weight auto] epoch 9: 17.2s, IoU 0.273
  [VOC 사람 256, pos_weight auto] epoch 10: 17.2s, IoU 0.280
=== VOC 256 (Tesla T4) ===


,설정,입력,학습 장 수,풀링,파라미터 수,1에폭 시간(초),"에폭당 평균(초, 1에폭 제외)",에폭,최종 IoU,최고 IoU(에폭),대상 없는 사진 오탐,장당 추론(ms)
0,"VOC 사람 256, pos_weight auto",256x256,1464,3,488705,17.1,17.3,10,0.28,0.280 (10),4.35%,5.39


In [12]:
print("GPU:", GPU_NAME)
for nm in ["df_a", "df_b"]:
    if nm in globals():
        print(f"\n### {nm}"); print(globals()[nm].to_string(index=False))

GPU: Tesla T4

### df_a
                설정      입력  학습 장 수  풀링  파라미터 수  1에폭 시간(초)  에폭당 평균(초, 1에폭 제외)  에폭  최종 IoU 최고 IoU(에폭) 대상 없는 사진 오탐  장당 추론(ms)
     Pet 256, 400장 256x256     400   3  488705        4.8                4.8   5   0.338  0.445 (4)        None       5.41
     Pet 256, 842장 256x256     842   3  488705       10.1                9.9   5   0.506  0.506 (5)        None       5.38
       Pet 256, 전부 256x256    3680   3  488705       43.5               43.4   5   0.642  0.642 (5)        None       5.31
Pet 256, 전부, 풀링 4번 256x256    3680   4 1965569       46.6               46.6   5   0.670  0.670 (5)        None       6.55

### df_b
                         설정      입력  학습 장 수  풀링  파라미터 수  1에폭 시간(초)  에폭당 평균(초, 1에폭 제외)  에폭  최종 IoU 최고 IoU(에폭) 대상 없는 사진 오탐  장당 추론(ms)
VOC 사람 256, pos_weight auto 256x256    1464   3  488705       17.1               17.3  10    0.28 0.280 (10)       4.35%       5.39
